# Cursul 3. Surse diverse și preprocesare
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

În C2 am adus datele prin API. Azi mutăm funcțiile într-un fișier `.py`, aducem mai mulți indicatori și pregătim tabelele ca să le putem uni în C4: nume clare pentru coloane și doar rândurile de care avem nevoie. La final, echipa își aduce depozitul proiectului.

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Funcțiile din C2, într-un fișier `.py` | 10 min |
| 2 | Ce am adus din C2 | 5 min |
| 3 | Numele regiunilor: nomenclatorul NUTS | 10 min |
| 4 | Mai mulți indicatori regionali, într-un singur tabel | 15 min |
| 5 | SCI pe regiuni: nume clare | 10 min |
| 6 | Perechi de țări: migrația și zborurile | 20 min |
| 7 | Echipa: depozitul proiectului | 20 min |

Fiecare pas salvează un fișier în `data/c3`. De la ele pornește C4.

## 1. Funcțiile din C2, într-un fișier `.py`
În C2 am scris `cere_bm` și `cere_eurostat` în notebook. Ca să le folosim în orice notebook, și mai târziu în proiect, le punem într-un fișier de cod: `src/date_api.py`, ca în proiectul echipei. Descarcă-l din depozitul cursului în folderul `curs/src/` și deschide-l în VS Code: sunt aceleași funcții, plus `cere_perechi`, bucla din C2 pentru perechi de țări.

`from src.date_api import ...` înseamnă: din fișierul `date_api.py` din folderul `src`, ia funcțiile numite.

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from src.date_api import cere_bm, cere_eurostat, cere_perechi   # funcțiile din src/date_api.py

# Folderul pentru fișierele de azi (dacă există deja, nu se întâmplă nimic)
Path("data/c3").mkdir(exist_ok=True)

Funcția se folosește exact ca în C2: densitatea tuturor regiunilor NUTS2, într-un rând.

In [ ]:
cere_eurostat("demo_r_d3dens", {"geoLevel": "nuts2", "time": 2024}).head()

Dacă schimbi ceva în `src/date_api.py`, repornește kernelul (**Restart**): Python citește fișierul o singură dată, la primul `import`.

## 2. Ce am adus din C2
În `data/c2` avem câte un fișier pe indicator și nivel. **Anticipează:** ce coloane are unul dintre ele?

In [ ]:
densitate = pd.read_csv("data/c2/densitate_nuts3.csv")    # densitatea pe județe, din C2
densitate.head()

Două coloane: codul regiunii (`geo`) și valoarea. Un rând = o regiune. Clujul îl găsim după cod, cu filtrul din C1:

In [ ]:
densitate[densitate["geo"] == "RO113"]      # rândurile în care geo este RO113

**Exercițiu.** Câți locuitori are regiunea Nord-Vest (`RO11`)? Fișierul este `populatie_nuts2.csv`.

In [ ]:
# 1. Citește data/c2/populatie_nuts2.csv în variabila populatie
# 2. Rândul în care geo este "RO11", ca la densitate

Codul `RO11` nu spune singur că e vorba de Nord-Vest. Numele vin din nomenclatorul NUTS.

## 3. Numele regiunilor: nomenclatorul NUTS
Eurostat publică lista tuturor regiunilor (codul, țara, numele) ca fișier CSV, pe serverul GISCO ([pagina fișierelor](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)). Îl descărcăm cu `requests` și îl salvăm în `data/c3`.

In [ ]:
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/csv/NUTS_AT_2024.csv"
raspuns = requests.get(url)                                       # cerem fișierul, ca în C2
fisier = Path("data/c3/nuts_gisco.csv")
fisier.write_bytes(raspuns.content)                               # îl salvăm exact cum a venit

nuts = pd.read_csv(fisier)                                        # și îl citim
nuts.head()

Păstrăm trei coloane și le dăm nume în română, aceleași în tot proiectul. Lungimea codului spune nivelul: `RO` e țara, `RO11` regiunea (NUTS2), `RO113` județul (NUTS3).

In [ ]:
nuts = nuts[["NUTS_ID", "CNTR_CODE", "NAME_LATN"]]   # codul, țara, numele cu litere latine
nuts.columns = ["regiune", "tara", "denumire"]       # nume noi, în aceeași ordine
nuts["nivel"] = nuts["regiune"].str.len() - 2        # RO -> 0, RO11 -> 2, RO113 -> 3
nuts[nuts["tara"] == "RO"].head(8)

**Exercițiu.** Care sunt regiunile NUTS2 ale României? Două condiții, fiecare între paranteze, legate cu `&`.

In [ ]:
# Rândurile din nuts în care tara este "RO" și nivel este 2

In [ ]:
nuts.to_csv("data/c3/nuts.csv", index=False)

## 4. Mai mulți indicatori regionali, într-un singur tabel
Din C2 avem populația, densitatea și PIB-ul pe locuitor. Literatura despre SCI arată că legăturile depind și de cât de asemănătoare sunt două regiuni (vârstă, educație, piața muncii) și de cât folosesc oamenii rețelele sociale. Le aducem din Eurostat; codurile și filtrele sunt în `data/CATALOG.md`.

Întâi cele trei fișiere din C2. Pentru fiecare adăugăm două coloane, ca să știm de unde vine fiecare rând: ce măsoară și la ce nivel.

In [ ]:
def citeste(indicator, nivel):
    # Un fișier din C2, cu două coloane în plus: ce măsoară și la ce nivel
    tabel = pd.read_csv(f"data/c2/{indicator}_nuts{nivel}.csv")   # de exemplu data/c2/densitate_nuts3.csv
    tabel["indicator"] = indicator                              # ce măsoară valoarea
    tabel["nivel"] = nivel                                      # 2 = regiuni, 3 = județe
    return tabel.rename(columns={"geo": "regiune"})            # același nume ca în nomenclator

din_c2 = [citeste("populatie", 2), citeste("populatie", 3),
          citeste("densitate", 2), citeste("densitate", 3),
          citeste("pib_locuitor", 2), citeste("pib_locuitor", 3)]

Acum indicatorii noi, pe regiuni NUTS2. Fiecare are setul lui și filtrele lui, ca în C2: numele nostru → (setul Eurostat, filtrele).

In [ ]:
NOI = {
    "varsta_medie":      ("demo_r_pjanind2", {"indic_de": "MEDAGEPOP", "unit": "YR"}),
    "studii_superioare": ("edat_lfse_04",    {"isced11": "ED5-8", "age": "Y25-64", "sex": "T", "unit": "PC"}),
    "somaj":             ("lfst_r_lfu3rt",   {"isced11": "TOTAL", "age": "Y15-74", "sex": "T", "unit": "PC"}),
    "retele_sociale":    ("isoc_r_iuse_i",   {"indic_is": "I_IUSNET", "unit": "PC_IND"}),
}

noi = []
for nume, (dataset, filtre) in NOI.items():
    tabel = cere_eurostat(dataset, {**filtre, "geoLevel": "nuts2", "time": 2024})   # toate regiunile NUTS2
    tabel["indicator"] = nume
    tabel["nivel"] = 2
    noi.append(tabel.rename(columns={"geo": "regiune"}))
    print(nume, len(tabel), "regiuni")

Le punem pe toate unul sub altul, cu `pd.concat`, ca la Banca Mondială în C2.

In [ ]:
regional = pd.concat(din_c2 + noi)      # cele 6 tabele din C2 și cele 4 noi
regional.shape                          # rânduri, coloane

Tabelul e în format **lung**: o regiune apare de mai multe ori, câte un rând pe indicator.

**Exercițiu.** Afișează toate rândurile regiunii Nord-Vest (`RO11`).

In [ ]:
# Rândurile din regional în care regiune este "RO11"

**Verificăm:** câte rânduri are fiecare indicator și câte au valoare? `size` numără rândurile, `count` doar valorile prezente; diferența sunt valorile lipsă.

In [ ]:
regional.groupby(["nivel", "indicator"])["valoare"].agg(["size", "count"])

**Interpretează:** unde lipsesc cele mai multe valori? O valoare lipsă nu înseamnă zero: sursa nu are date pentru acea regiune în 2024. Ce facem cu ele decidem în C4.

In [ ]:
regional.to_csv("data/c3/regional.csv", index=False)

## 5. SCI pe regiuni: nume clare
SCI există și pe regiuni NUTS2, din aceeași sursă ca `country.csv` din C1 ([HDX](https://data.humdata.org/dataset/social-connectedness-index)). Fișierul stă în depozitul cursului; îl aducem cu `requests`, ca nomenclatorul.

In [ ]:
url = "https://raw.githubusercontent.com/dsw-ubb-2026/-DSW2026/main/curs/data/nuts2_2024.csv"
raspuns = requests.get(url)                       # cerem fișierul, ca nomenclatorul
fisier = Path("data/nuts2_2024.csv")
fisier.write_bytes(raspuns.content)               # îl salvăm exact cum a venit

sci = pd.read_csv(fisier)
sci.head()

Coloanele au nume în engleză. Le dăm nume care spun ce înseamnă în proiect: cele două capete ale perechii.

In [ ]:
nume_noi = {"user_country": "tara_1", "friend_country": "tara_2",
            "user_region": "regiune_1", "friend_region": "regiune_2",
            "scaled_sci": "sci"}
sci = sci.rename(columns=nume_noi)      # se schimbă doar numele din dicționar
print(sci["regiune_1"].nunique(), "regiuni")
print(len(sci), "rânduri")

298 × 298 = 88.804: fiecare regiune cu fiecare, inclusiv cu ea însăși.

**Exercițiu.** Cât este SCI între Nord-Vest (`RO11`) și Lombardia (`ITC4`)? Apoi schimbă ordinea celor două coduri.

In [ ]:
# Rândul din sci în care regiune_1 este "RO11" și regiune_2 este "ITC4"

Aceeași valoare, în ambele ordini: fiecare pereche apare de două ori. În C4 păstrăm o singură apariție.

In [ ]:
sci.to_csv("data/c3/sci_nuts2.csv", index=False)

SCI pe județe (NUTS3) are 1,8 milioane de rânduri, peste 40 MB. Nu îl folosim la curs; echipele care au nevoie de el îl descarcă de pe [HDX](https://data.humdata.org/dataset/social-connectedness-index) și fac aceiași pași.

## 6. Perechi de țări: migrația și zborurile
### 6.1. Migrația: țări, nu grupuri de țări
Deschidem tabelul din C2. O capcană: codul Namibiei este `NA`, iar pandas citește `NA` ca valoare lipsă. Îi spunem să considere lipsă doar celulele goale.

In [ ]:
migratie = pd.read_csv("data/c2/migratie.csv", keep_default_na=False, na_values=[""])
migratie["citizen"].unique()

**Anticipează:** care coduri nu sunt țări? Eurostat pune în aceeași coloană și grupuri: `EU27_2020_FOR` (străinii din UE), `FOR` (toți străinii), `NAT` (cetățenii proprii), `TOTAL`. Dacă le păstrăm, aceiași oameni sunt numărați de mai multe ori. Codurile de țară au exact două litere; păstrăm rândurile în care **ambele** coloane sunt țări.

In [ ]:
cetatenie_tara = migratie["citizen"].str.len() == 2     # True unde cetățenia e o țară
rezidenta_tara = migratie["geo"].str.len() == 2         # True unde locul e o țară
migratie = migratie[cetatenie_tara & rezidenta_tara]    # amândouă

Scoatem și cetățenii propriei țări (românii din România): nu sunt migranți. Apoi păstrăm trei coloane, cu nume clare.

In [ ]:
migratie = migratie[migratie["citizen"] != migratie["geo"]]      # cetățenia diferă de țară
migratie = migratie[["geo", "citizen", "valoare"]]               # coloanele, în ordinea dorită
migratie.columns = ["tara_rezidenta", "cetatenie", "persoane"]   # nume clare
migratie.to_csv("data/c3/migratie.csv", index=False)
migratie[migratie["cetatenie"] == "RO"].sort_values("persoane", ascending=False).head(5)

### 6.2. Zborurile: o legătură vie între țări
Migrația spune câți oameni s-au mutat. Pasagerii aerieni dintre două țări arată cât se circulă între ele: vizite, întoarceri, du-te-vino. Setul este `avia_paocc` (vezi `data/CATALOG.md`).

Ca la migrație, într-o cerere variază un singur lucru, deci cerem țară cu țară. `cere_perechi` face bucla din C2: fixează pe rând fiecare țară (`geo`) și ia toate țările partenere (`pe="partner"`). Ca listă de țări folosim țările din tabelul migrației.

In [ ]:
tari = sorted(migratie["tara_rezidenta"].unique())             # țările Eurostat
filtre = {"unit": "PAS", "tra_meas": "PAS_CRD", "schedule": "TOTAL", "freq": "A", "time": 2024}

zboruri = cere_perechi("avia_paocc", filtre, tari, pe="partner")   # o cerere pe țară
zboruri.head()

Aceeași curățenie ca la migrație: doar țări, fără perechea unei țări cu ea însăși, cu nume clare.

In [ ]:
zboruri = zboruri[zboruri["partner"].str.len() == 2]             # doar țări, nu grupuri (EU27_2020)
zboruri = zboruri[zboruri["partner"] != zboruri["geo"]]          # fără zborurile interne
zboruri = zboruri[["geo", "partner", "valoare"]]
zboruri.columns = ["tara_1", "tara_2", "pasageri"]
zboruri.to_csv("data/c3/zboruri.csv", index=False)

**Exercițiu.** Cu ce țări are România cei mai mulți pasageri? Primele 10. Unde e Italia, comparativ cu migrația și cu SCI din C1?

In [ ]:
# Rândurile din zboruri în care tara_1 este "RO", ordonate după pasageri, primele 10

Fiecare țară raportează propriile zboruri, deci RO-IT și IT-RO apar separat, cu valori apropiate. În C4 alegem una dintre ele, ca la SCI.

Alte date pe perechi de țări (turiști, comerț, imigranți pe an) sunt în `data/CATALOG.md`, cu filtrele lor. Le aduceți cu aceleași funcții.

## 7. Echipa: depozitul proiectului
De azi lucrați și în echipă. Fiecare echipă are pe GitHub depozitul ei, `dsw-ubb-2026/echipa-N`, cu aceeași structură pentru toate echipele. Invitația a venit pe emailul contului GitHub.

### 7.1. Depozitul pe calculator (fiecare membru)
1. Acceptați invitația: `https://github.com/dsw-ubb-2026/echipa-N/invitations` (N = numărul echipei).
2. În terminalul din VS Code, în folderul `dsw/` (nu în `curs/`):
```bash
git clone https://github.com/dsw-ubb-2026/echipa-N.git proiect
```
Prima dată pe un calculator, Git vă cere numele și emailul:
```bash
git config --global user.name "Prenume Nume"
git config --global user.email "emailul-de-pe-github"
```
Rezultatul:
```text
dsw/
├── .venv/
├── curs/                 caietele de curs și src/date_api.py, fără Git
└── proiect/              depozitul echipei, cu Git
    ├── data/raw/         datele originale, neschimbate
    ├── data/processed/   tabelele produse de caiete
    ├── notebooks/        câte un caiet pe pas
    ├── src/              cod comun, în fișiere .py
    ├── figures/          figurile pentru raport
    ├── docs/             raportul
    └── README.md         descrierea proiectului
```

### 7.2. Datele și codul proiectului
Datele de la curs rămân în `curs/data`. Datele proiectului stau în `proiect/data`: în `raw/` ce aduceți de la sursă, exact cum a venit; în `processed/` ce produc caietele voastre. Fișierele de peste 50 MB nu se urcă pe GitHub: caietul care le folosește spune de unde se descarcă.

Copiați `curs/src/date_api.py` în `proiect/src/`: e codul comun al echipei, același ca la curs. Într-un caiet din `proiect/notebooks/` îl importați cu:
```python
import sys
sys.path.append("..")                 # folderul proiectului
from src.date_api import cere_eurostat
```
Ce seturi de date puteți folosi, cu filtrele lor: `curs/data/CATALOG.md`.

**Un proiect-model**, cu exact această structură, apare după curs în depozitul cursului, în `proiect-exemplu/`: `notebooks/01_collection.ipynb` aduce toate datele în `data/raw/` cu funcțiile din `src/date_api.py`, iar `02_preprocessing.ipynb` face din ele tabelele din `data/processed/`. Așa arată pașii 1 și 2 din rolurile echipei.

### 7.3. Primul commit al echipei
Un singur membru completează în `README.md`, provizoriu: titlul, întrebarea de cercetare, unitatea de analiză și tabelul surselor de date. Altul adaugă `src/date_api.py` (`git add src/date_api.py`). Fiecare urcă ce a făcut:
```bash
cd proiect
git pull
git add README.md
git commit -m "Tema si intrebarea provizorie"
git push
```
`commit` salvează pasul pe calculatorul vostru, `push` îl trimite pe GitHub, `pull` îl aduce de la colegi. Regula care evită conflictele: fiecare lucrează în fișierul lui și dă `git pull` înainte să înceapă.

### 7.4. Ce stabiliți azi în echipă
- tema proiectului și o întrebare provizorie: coridorul sau țara pe care o studiați;
- un rol provizoriu pentru fiecare: colectarea, preprocesarea, curățarea și integrarea, explorarea, modelul. Rolurile se pot schimba, iar toți pot colabora la orice pas;
- ce date din catalog vă ajută să răspundeți la întrebare;
- ce reprezintă un rând în tabelul vostru final.

Tema 1, pe Teams, cere fiecăruia să descrie cu propriile cuvinte tema proiectului, datele și rolul ales.

## Ce ducem mai departe
În `data/c3` avem `nuts.csv`, `regional.csv` (șapte indicatori), `sci_nuts2.csv`, `migratie.csv` și `zboruri.csv`: nume clare de coloane, gata de unit în C4. Funcțiile de colectare stau în `src/date_api.py`, iar echipa are depozitul pe calculator și primul commit pe GitHub.

**Poți explica fără cod?** De ce punem funcțiile într-un fișier `.py`, în `src/`? Ce reprezintă un rând în fiecare tabel? De ce adăugăm coloanele `indicator` și `nivel`? De ce `NA` nu e întotdeauna o valoare lipsă? De ce scoatem grupurile de țări? Ce diferență este între `commit`, `push` și `pull`?

### Surse
- [GISCO: fișierele NUTS 2024](https://gisco-services.ec.europa.eu/distribution/v2/nuts/nuts-2024-files.html)
- [Social Connectedness Index, pe HDX](https://data.humdata.org/dataset/social-connectedness-index)
- [Eurostat Data Browser](https://ec.europa.eu/eurostat/databrowser/): `migr_pop1ctz`, `avia_paocc`, `demo_r_pjanind2`, `edat_lfse_04`, `lfst_r_lfu3rt`, `isoc_r_iuse_i`
- Bailey, Johnston, Kuchler, Russel, State și Stroebel (2020), *The Determinants of Social Connectedness in Europe*, SocInfo.
- [Git: înregistrarea schimbărilor](https://git-scm.com/book/en/v2/Git-Basics-Recording-Changes-to-the-Repository)